In [ ]:
import pandas as pd
import numpy as np
from statsmodels.formula.api import ols 
from math import pi
import matplotlib.pyplot as plt
from arch import arch_model


# WIG20 5-minute data


- the "total_data_shifted" csv file constains data related to polish wig20 index : 5-minute returns (open, high,low,close ) , volume , etc. 

In [ ]:
fdt = pd.read_csv("total_data_shifted.csv", sep=";") 


In [3]:
display(fdt)

,TICKER,PER,DATE,TIME,OPEN,HIGH,LOW,CLOSE,VOL,OPENINT
0,WIG20,5,20250103,90000,2229.21,2233.88,2223.07,2223.07,139734.0,0
1,WIG20,5,20250103,90500,2222.55,2225.92,2222.18,2225.92,135215.0,0
2,WIG20,5,20250103,91000,2226.48,2234.67,2226.48,2234.67,97970.0,0
3,WIG20,5,20250103,91500,2234.51,2237.37,2232.85,2234.50,98568.0,0
4,WIG20,5,20250103,92000,2234.43,2234.43,2230.74,2230.74,135657.0,0
...,...,...,...,...,...,...,...,...,...,...
28370,WIG20,5,20260319,162500,3276.35,3277.09,3273.0,3274.68,458386.0,0
28371,WIG20,5,20260319,163000,3274.89,3274.89,3272.26,3273.87,413634.0,0
28372,WIG20,5,20260319,163500,3273.55,3274.94,3273.41,3274.61,513739.0,0
28373,WIG20,5,20260319,164000,3275.25,3276.35,3272.53,3274.47,460577.0,0


In [7]:
fdt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28375 entries, 0 to 28374
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   DATE     28375 non-null  int64         
 1   TIME     28375 non-null  int32         
 2   CLOSE    28375 non-null  float64       
 3   OPEN     28375 non-null  float64       
 4   TIME_dt  28375 non-null  datetime64[ns]
 5   session  28375 non-null  object        
dtypes: datetime64[ns](1), float64(2), int32(1), int64(1), object(1)
memory usage: 1.2+ MB


In [8]:
fdt.dtypes

DATE                int64
TIME                int32
CLOSE             float64
OPEN              float64
TIME_dt    datetime64[ns]
session            object
dtype: object

## necessary changes to be suitable

In [ ]:


fdt = fdt.loc[:, ['DATE', 'TIME', 'CLOSE', 'OPEN']].copy() # The dataframe is filtered to include only the relevant columns: 'DATE', 'TIME', 'CLOSE', and 'OPEN'.


fdt['TIME'] = fdt['TIME'].astype(str).str.zfill(6).astype(int)

fdt["TIME_dt"] = pd.to_datetime(fdt["TIME"].astype(str), format="%H%M%S")




fdt["TIME_dt"] = fdt["TIME_dt"] + pd.Timedelta(minutes=5) # shifting data by 5 minutes is not necessary for other user launches.

fdt["TIME"] = fdt["TIME_dt"].dt.strftime("%H%M%S").astype(int)


fdt['session'] = pd.to_datetime(fdt['TIME'].astype( str).str.zfill(6), format='%H%M%S').dt.time






,DATE,TIME,CLOSE,OPEN,TIME_dt,session
0,20250103,91500,2223.07,2229.21,1900-01-01 09:15:00,09:15:00
1,20250103,92000,2225.92,2222.55,1900-01-01 09:20:00,09:20:00
2,20250103,92500,2234.67,2226.48,1900-01-01 09:25:00,09:25:00
3,20250103,93000,2234.50,2234.51,1900-01-01 09:30:00,09:30:00
4,20250103,93500,2230.74,2234.43,1900-01-01 09:35:00,09:35:00
...,...,...,...,...,...,...
28370,20260319,164000,3274.68,3276.35,1900-01-01 16:40:00,16:40:00
28371,20260319,164500,3273.87,3274.89,1900-01-01 16:45:00,16:45:00
28372,20260319,165000,3274.61,3273.55,1900-01-01 16:50:00,16:50:00
28373,20260319,165500,3274.47,3275.25,1900-01-01 16:55:00,16:55:00


# the final shape of the dataframe which is applicable to the function

In [9]:
display(fdt)

,DATE,TIME,CLOSE,OPEN,TIME_dt,session
0,20250103,91500,2223.07,2229.21,1900-01-01 09:15:00,09:15:00
1,20250103,92000,2225.92,2222.55,1900-01-01 09:20:00,09:20:00
2,20250103,92500,2234.67,2226.48,1900-01-01 09:25:00,09:25:00
3,20250103,93000,2234.50,2234.51,1900-01-01 09:30:00,09:30:00
4,20250103,93500,2230.74,2234.43,1900-01-01 09:35:00,09:35:00
...,...,...,...,...,...,...
28370,20260319,164000,3274.68,3276.35,1900-01-01 16:40:00,16:40:00
28371,20260319,164500,3273.87,3274.89,1900-01-01 16:45:00,16:45:00
28372,20260319,165000,3274.61,3273.55,1900-01-01 16:50:00,16:50:00
28373,20260319,165500,3274.47,3275.25,1900-01-01 16:55:00,16:55:00


In [10]:
fdt.dtypes

DATE                int64
TIME                int32
CLOSE             float64
OPEN              float64
TIME_dt    datetime64[ns]
session            object
dtype: object